# SoM Agent 错误案例可视化（click / long_press）

在下方修改 **全局配置** 后依次运行各 cell。

图层说明：
- **绿色虚线框**：GT `nearest_5` 可接受节点
- **青色实线框**：检索 top-k 候选（带 `#node_id`）
- **红色粗框**：VLM 选择的 `node_id`
- **绿色十字**：GT 点击坐标 `(x, y)`

分两页展示（各用 `PAGE` 分页）：
1. **Top-K 未命中**：`retrieval_hit == false` 的错误步
2. **Top-K 命中 / VLM 选错**：`retrieval_hit == true` 但 `step_correct == false`

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from PIL import Image

# ========== 全局配置 ==========
AC_MODE = "low"           # "low" | "high"
AGENT = "m2"              # "m2" | "m2p"
MODEL = "qwen-vl-max"     # 与 runs 文件名末尾 vlm_model 一致
TOP_K = 5
TO_SELECT = "generate"    # generate | best | mid | worst

COLS = 4                  # 每页列数
ROWS = 3                  # 每页行数
PAGE = 0                  # 从 0 起，步数多时分页查看
# ==============================

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from eval.run_naming import run_filename
from eval.step_judge import compute_retrieval_hit
from process.paths import step_paths

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

POINTER_TYPES = {"click", "long_press", "long_click"}
RUN_PATH = PROJECT_ROOT / "runs" / run_filename(AC_MODE, AGENT, TOP_K, MODEL, to_select=TO_SELECT)
print(f"项目根目录: {PROJECT_ROOT}")
print(f"Run 文件: {RUN_PATH}")
assert RUN_PATH.is_file(), f"找不到 run 文件: {RUN_PATH}"

In [ ]:
def _norm_action_type(action_type: str | None) -> str:
    if action_type == "long_click":
        return "long_press"
    return action_type or ""


def _retrieval_hit(step: dict) -> bool | None:
    if "retrieval_hit" in step:
        hit = step.get("retrieval_hit")
        return None if hit is None else bool(hit)
    return compute_retrieval_hit(
        step.get("gt") or {},
        step.get("top_k_nodes"),
        top_k=TOP_K,
    )


def load_wrong_pointer_steps(run_path: Path) -> tuple[list[dict], list[dict]]:
    """收集 click/long_press 错误步，并按 retrieval 是否命中拆分。"""
    with open(run_path, "r", encoding="utf-8") as f:
        episodes = json.load(f)

    retrieval_miss: list[dict] = []
    retrieval_hit_vlm_wrong: list[dict] = []

    for ep in episodes:
        for step in ep.get("steps", []):
            if step.get("status") == "skipped":
                continue
            gt = step.get("gt") or {}
            gt_type = _norm_action_type(gt.get("action_type"))
            if gt_type not in {"click", "long_press"}:
                continue
            if step.get("step_correct", True):
                continue

            record = {
                "episode_id": ep.get("episode_id"),
                "goal": ep.get("goal", ""),
                **step,
                "gt_type_norm": gt_type,
            }
            rh = _retrieval_hit(record)
            if rh is False:
                retrieval_miss.append(record)
            elif rh is True:
                retrieval_hit_vlm_wrong.append(record)

    return retrieval_miss, retrieval_hit_vlm_wrong


def _bounds_center(bounds: list) -> tuple[float, float] | None:
    if len(bounds) != 4:
        return None
    x1, y1, x2, y2 = bounds
    if x2 <= x1 or y2 <= y1:
        return None
    return (x1 + x2) / 2.0, (y1 + y2) / 2.0


def _load_node_bounds(stem: str, node_id: int) -> list | None:
    nodes_path = step_paths(stem)["nodes"]
    if not nodes_path.is_file():
        return None
    with open(nodes_path, "r", encoding="utf-8") as f:
        nodes = json.load(f)
    for node in nodes:
        if int(node.get("node_id", -1)) == int(node_id):
            b = node.get("bounds")
            if isinstance(b, list) and len(b) == 4:
                return b
    return None


def _pred_node_bounds(step: dict) -> tuple[int | None, list | None]:
    pred = step.get("pred_action") or {}
    node_id = pred.get("node_id")
    if node_id is None:
        return None, None
    node_id = int(node_id)
    for n in step.get("top_k_nodes") or []:
        if int(n.get("node_id", -1)) == node_id:
            return node_id, n.get("bounds")
    bounds = _load_node_bounds(step["stem"], node_id)
    return node_id, bounds


def _error_label(step: dict) -> str:
    pred_type = _norm_action_type((step.get("pred_action") or {}).get("action_type"))
    gt_type = step.get("gt_type_norm", "")
    if not step.get("type_correct", True):
        return f"type: {pred_type}≠{gt_type}"
    detail = step.get("eval_detail") or {}
    pred_id, _ = _pred_node_bounds(step)
    gt_ids = [int(x["node_id"]) for x in (step.get("gt") or {}).get("nearest_5", []) if "node_id" in x]
    rh = _retrieval_hit(step)
    parts = [f"pred #{pred_id}" if pred_id is not None else "pred ?"]
    if gt_ids:
        parts.append(f"GT {gt_ids}")
    if rh is not None:
        parts.append(f"retrieval={'hit' if rh else 'miss'}")
    if detail.get("norm_distance") is not None:
        parts.append(f"dist={detail['norm_distance']:.3f}")
    return " | ".join(parts)


retrieval_miss, retrieval_hit_vlm_wrong = load_wrong_pointer_steps(RUN_PATH)
wrongs = retrieval_miss + retrieval_hit_vlm_wrong
print(f"{AC_MODE} | {AGENT} | top{TOP_K}{TO_SELECT} | {MODEL}")
print(f"click/long_press 错误步数: {len(wrongs)}")
print(f"  - Top-K 未命中: {len(retrieval_miss)}")
print(f"  - Top-K 命中 / VLM 选错: {len(retrieval_hit_vlm_wrong)}")

In [ ]:
def draw_wrong_step(ax, step: dict) -> None:
    stem = step["stem"]
    screenshot = step_paths(stem)["screenshot"]
    if not screenshot.is_file():
        ax.text(0.5, 0.5, f"缺图\n{stem}", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")
        return

    img = Image.open(screenshot).convert("RGB")
    ax.imshow(img)

    gt = step.get("gt") or {}
    nearest_5 = gt.get("nearest_5") or []
    top_k = step.get("top_k_nodes") or []
    pred_id, pred_bounds = _pred_node_bounds(step)
    top_k_ids = {int(n["node_id"]) for n in top_k if "node_id" in n}
    gt_ids = {int(n["node_id"]) for n in nearest_5 if "node_id" in n}

    # 1) GT nearest_5 — 绿色虚线
    for item in nearest_5:
        b = item.get("bounds")
        if not b or len(b) != 4:
            continue
        x1, y1, x2, y2 = b
        nid = item.get("node_id")
        ax.add_patch(
            patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2.5, edgecolor="#2ecc71", facecolor="none", linestyle="--",
            )
        )
        ax.text(x1 + 4, y1 + 4, f"GT#{nid}", color="#1e8449", fontsize=7,
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.75, edgecolor="none"))

    # 2) 检索 top-k — 青色（跳过与 GT 完全重合的框可仍画，用不同色区分）
    for node in top_k:
        b = node.get("bounds")
        if not b or len(b) != 4:
            continue
        nid = int(node["node_id"])
        if pred_id is not None and nid == pred_id:
            continue  # pred 单独用红色画
        x1, y1, x2, y2 = b
        ax.add_patch(
            patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=1.8, edgecolor="#3498db", facecolor=(52/255, 152/255, 219/255, 0.12),
            )
        )
        ax.text(x1 + 4, max(y1 - 2, 0), f"#{nid}", color="#1a5276", fontsize=7,
                va="bottom", bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.7, edgecolor="none"))

    # 3) VLM pred — 红色粗框
    if pred_bounds and len(pred_bounds) == 4:
        x1, y1, x2, y2 = pred_bounds
        ax.add_patch(
            patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=3.5, edgecolor="#e74c3c", facecolor=(231/255, 76/255, 60/255, 0.18),
            )
        )
        ax.text(x2 - 4, y2 - 4, f"pred #{pred_id}", color="#922b21", fontsize=8,
                ha="right", va="top", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="#fadbd8", alpha=0.9, edgecolor="none"))
    elif pred_id is not None:
        ax.text(8, 24, f"pred #{pred_id} (无 bbox)", color="#e74c3c", fontsize=8,
                bbox=dict(facecolor="white", alpha=0.8))

    # 4) GT 坐标十字
    if "x" in gt and "y" in gt:
        gx, gy = float(gt["x"]), float(gt["y"])
        ax.scatter([gx], [gy], s=120, c="#27ae60", marker="+", linewidths=2.5, zorder=10)
        ax.scatter([gx], [gy], s=40, c="#27ae60", marker="o", facecolors="none", linewidths=1.5, zorder=10)

    instr = (step.get("display_instruction") or step.get("instruction") or "")[:48]
    title = f"{stem}\n{instr}"
    if len((step.get("display_instruction") or step.get("instruction") or "")) > 48:
        title += "…"
    ax.set_title(title, fontsize=8)
    ax.text(
        0.01, 0.99, _error_label(step), transform=ax.transAxes,
        fontsize=7, va="top", color="#2c3e50",
        bbox=dict(boxstyle="round,pad=0.25", facecolor="#fef9e7", alpha=0.92, edgecolor="#f4d03f"),
    )
    ax.axis("off")


def plot_wrong_cases(
    cases: list[dict],
    *,
    category: str,
    page: int = 0,
    cols: int = 4,
    rows: int = 6,
) -> None:
    per_page = cols * rows
    start = page * per_page
    end = min(start + per_page, len(cases))
    chunk = cases[start:end]
    if not chunk:
        print(f"[{category}] 第 {page} 页无数据（共 {len(cases)} 条）")
        return

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 5.8))
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for ax, step in zip(axes_flat, chunk):
        draw_wrong_step(ax, step)
    for ax in axes_flat[len(chunk):]:
        ax.axis("off")

    legend_elems = [
        Line2D([0], [0], color="#2ecc71", lw=2.5, linestyle="--", label="GT nearest_5"),
        patches.Patch(facecolor=(52/255, 152/255, 219/255, 0.3), edgecolor="#3498db", label="检索 top-k"),
        patches.Patch(facecolor=(231/255, 76/255, 60/255, 0.35), edgecolor="#e74c3c", linewidth=2, label="VLM pred"),
        Line2D([0], [0], marker="+", color="#27ae60", lw=0, markersize=10, label="GT (x,y)"),
    ]
    fig.legend(handles=legend_elems, loc="lower center", ncol=4, fontsize=9, frameon=True)
    fig.suptitle(
        f"{AC_MODE} | {AGENT} | top{TOP_K}{TO_SELECT} | {MODEL} — {category} {start + 1}-{end} / {len(cases)}",
        fontsize=12, y=1.01,
    )
    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    plt.show()

In [ ]:
plot_wrong_cases(retrieval_miss, category="Top-K 未命中", page=PAGE, cols=COLS, rows=ROWS)
plot_wrong_cases(retrieval_hit_vlm_wrong, category="Top-K 命中 / VLM 选错", page=PAGE, cols=COLS, rows=ROWS)

In [ ]:
# 可选：多 agent 同页对比（需各自 run 文件存在）
COMPARE_AGENTS = ["m2", "m2p"]
COMPARE_STEM = None  # 例如 "00000140_005"；None 则取 m2 第一个错误步

def find_step_in_run(run_path: Path, stem: str) -> dict | None:
    with open(run_path, "r", encoding="utf-8") as f:
        for ep in json.load(f):
            for s in ep.get("steps", []):
                if s.get("stem") == stem:
                    gt = s.get("gt") or {}
                    return {**s, "gt_type_norm": _norm_action_type(gt.get("action_type"))}
    return None


if COMPARE_STEM is None and wrongs:
    COMPARE_STEM = wrongs[0]["stem"]

if COMPARE_STEM:
    fig, axes = plt.subplots(1, len(COMPARE_AGENTS), figsize=(5 * len(COMPARE_AGENTS), 6))
    if len(COMPARE_AGENTS) == 1:
        axes = [axes]
    for ax, agent in zip(axes, COMPARE_AGENTS):
        rp = PROJECT_ROOT / "runs" / run_filename(AC_MODE, agent, TOP_K, MODEL, to_select=TO_SELECT)
        step = find_step_in_run(rp, COMPARE_STEM) if rp.is_file() else None
        if step is None:
            ax.text(0.5, 0.5, f"{agent}\n无数据", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            continue
        draw_wrong_step(ax, step)
        sc = step.get("step_correct")
        ax.set_xlabel(f"{agent} | step={'✓' if sc else '✗'}", fontsize=10)
    fig.suptitle(f"同一步对比: {COMPARE_STEM}", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("无错误步可对比")